# 05 - Kiểm tra và đóng gói kết quả benchmark

Chạy **sau khi đã hoàn thành** ít nhất một notebook mô hình (01/02/03).

Kiểm tra tự động trên Drive:
1. Mỗi run có `metadata.json` + `run_log.csv` + đủ 29 dòng (success + failed = 29);
2. Mỗi ảnh success có file mesh tồn tại, đọc được, không rỗng;
3. Commit khớp với commit chuẩn của từng mô hình;
4. Xuất bảng tóm tắt + đường dẫn zip để tải về máy.

Notebook này chỉ kiểm tra/đóng gói — **không cần GPU**.

In [ ]:
!pip install -q trimesh

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, csv, json
import numpy as np
import trimesh

DRIVE_ROOT = '/content/drive/MyDrive/nien_luan_3d_benchmark'
RUNS_DIR = os.path.join(DRIVE_ROOT, 'runs')
DATASET_DIR = os.path.join(DRIVE_ROOT, 'dataset')

EXPECTED = {
    'triposr':      '107cefdc244c39106fa830359024f6a2f1c78871',
    'instantmesh':  '08822c52fdc399b93ea00e4fa9e596344ed52ccc',
    'hunyuan3d2mini': 'f8db63096c8282cb27354314d896feba5ba6ff8a',
}

manifest = {}
with open(os.path.join(DATASET_DIR, 'dataset_manifest.csv'), encoding='utf-8-sig') as f:
    for r in csv.DictReader(f):
        manifest[r['object_id']] = r
expected_ids = set(manifest.keys())
print('Dataset manifest:', len(expected_ids), 'objects')
print('RUNS_DIR:', RUNS_DIR)
print('Cac run tren Drive:', sorted(os.listdir(RUNS_DIR)) if os.path.exists(RUNS_DIR) else '(chua co)')

In [ ]:
def load_csv(path):
    with open(path, encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))

summary = []
for run_id in sorted(os.listdir(RUNS_DIR)):
    run_dir = os.path.join(RUNS_DIR, run_id)
    if not os.path.isdir(run_dir):
        continue
    rec = {'run_id': run_id, 'status': 'OK', 'details': []}
    meta_p = os.path.join(run_dir, 'metadata.json')
    log_p = os.path.join(run_dir, 'run_log.csv')
    mesh_dir = os.path.join(run_dir, 'meshes')

    if not (os.path.exists(meta_p) and os.path.exists(log_p)):
        rec['status'] = 'THIEU metadata/log'
        summary.append(rec)
        continue
    meta = json.load(open(meta_p, encoding='utf-8'))
    rows = load_csv(log_p)

    model = meta.get('model', run_id.split('_')[0])
    commit_ok = meta.get('commit') == EXPECTED.get(model)
    if not commit_ok:
        rec['details'].append('COMMIT SAI: %s (expect %s)' % (meta.get('commit'), EXPECTED.get(model)))
        rec['status'] = 'LOI'

    ids = set(r.get('object_id') for r in rows)
    if len(ids) != 29:
        rec['details'].append('Thieu object: %d/29' % len(ids))
        rec['status'] = 'LOI'
    missing_ids = expected_ids - ids
    if missing_ids:
        rec['details'].append('Chua co object: %s' % sorted(missing_ids)[:5])
        rec['status'] = 'LOI'
    if rec['status'] == 'OK':
        rec['status'] = 'CHUA DU' if len(ids) < 29 else 'OK'

    ok = sum(1 for r in rows if r.get('success') == 'True')
    fail = sum(1 for r in rows if r.get('success') == 'False')
    rec['success'] = ok
    rec['failed'] = fail
    rec['model'] = model
    rec['gpu'] = meta.get('gpu', '')
    rec['total_time'] = meta.get('batch_time_s', '')
    rec['zip_path'] = os.path.join(run_dir, run_id + '.zip')

    bad_mesh = []
    for r in rows:
        if r.get('success') != 'True':
            continue
        oid = r['object_id']
        glb = os.path.join(mesh_dir, oid + '.glb')
        if not os.path.exists(glb):
            bad_mesh.append((oid, 'thieu file'))
            continue
        try:
            m = trimesh.load(glb, process=False)
            if m.vertices.shape[0] == 0 or m.faces.shape[0] == 0:
                bad_mesh.append((oid, 'mesh rong'))
        except Exception as e:
            bad_mesh.append((oid, 'doc loi: %s' % type(e).__name__))
    if bad_mesh:
        rec['details'].append('Mesh loi: %s' % bad_mesh[:3])
        rec['status'] = 'LOI'
    rec['mesh_ok'] = len(bad_mesh)
    summary.append(rec)

print('%-30s %-6s %-5s %-5s %-8s %-16s %s' % ('run_id', 'model', 'ok', 'fail', 'meshLoi', 'gpu', 'status'))
for r in sorted(summary, key=lambda x: x['run_id']):
    print('%-30s %-6s %-5s %-5s %-8s %-16s %s' % (
        r['run_id'], r.get('model', '-'), r.get('success', '-'), r.get('failed', '-'),
        r.get('mesh_ok', '-'), r.get('gpu', '-')[:14], r['status']))
    for d in r.get('details', []):
        print('      ->', d)

In [ ]:
import zipfile

report = []
for r in summary:
    if r.get('zip_path') and os.path.exists(r['zip_path']):
        z = os.path.basename(r['zip_path'])
        with zipfile.ZipFile(r['zip_path']) as zf:
            names = zf.namelist()
        glb_count = sum(1 for n in names if n.endswith('.glb'))
        report.append((r['run_id'], z, os.path.getsize(r['zip_path']), glb_count))

print('Zip da dong goi (%d run):' % len(report))
for run_id, z, size, glb_count in report:
    print('  %-30s %-35s %10d bytes  (%d glb)' % (run_id, z, size, glb_count))

print()
print('HUONG DAN TAI VE MAY:')
print('1. Tren Colab trai: bieu tuong thu muc (folder) -> chon folder runs/<run_id>')
print('2. Nhan chuot phai file <run_id>.zip -> Download')
print('3. Giai nen vao thu muc: E:\\xampp\\htdocs\\3DGame\\experiments\\multimodel\\raw\\<run_id>')
print('4. Chay script kiem tra:  python -m scripts.benchmark.verify_multimodel_results')

## KẾT THÚC notebook 05

Gửi về:
1. Bảng tóm tắt (cell thứ 2);
2. Danh sách zip (cell thứ 3).

Sau khi tải về máy và chạy `verify_multimodel_results` thành công → bước đánh giá chất lượng bằng `evaluate_mesh.py` (cùng giao thức C1) sẽ được thực hiện ở bước tiếp theo.